# Курсовая работа по ЦТЗИ, вариант 22

Распознавание и синонимическое обезличивание персональных данных.

Запускайте ячейки сверху вниз в Google Colab. Повторное обучение и подключение Google Диска не нужны. Обученная RuBERT и токенизатор скачиваются из GitHub Releases. Если адрес репозитория ещё не настроен, ячейка загрузки попросит ввести его в формате `пользователь/репозиторий`.

Для ускорения распознавания и BERTScore выберите **Среда выполнения > Сменить среду выполнения > T4 GPU**. При отсутствии GPU программа использует CPU; полный расчёт займёт больше времени.

Для первого запуска нужен интернет: зависимости, обученная RuBERT, справочник КЛАДР и отдельная базовая модель BERTScore загружаются автоматически. BERTScore не обучает нашу модель распознавания.

Обязательные классы: NAME, SURNAME, INN, PASSPORT, CARD, OGRNIP, IP, EMAIL, BIRTH_CERT, KLADR. Границы `[тип, start, end]`, конец исключён. `anon(text)` возвращает преобразованный текст и список сущностей. `anonymize(text, entities)` работает по готовой разметке.

В распознавании используются RuBERT с `aggregation_strategy="first"` и правила для реквизитов. GLiNER не используется. Агрегирование здесь объединяет части распознанных слов. Телефон, СНИЛС, отчество и дата рождения не входят в десять оцениваемых типов.


In [ ]:
%pip install -q "transformers==4.57.6" "accelerate>=0.30,<2" "safetensors>=0.4,<1" "pymorphy3==2.0.6" "bert-score==0.3.13" "pandas>=2,<3"


In [ ]:
import re
import json
import hashlib
import random
import string
from pathlib import Path
from datetime import date

import torch
import pymorphy3
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

morph = pymorphy3.MorphAnalyzer()
rng = random.SystemRandom()
print("PyTorch:",torch.__version__)
print("GPU:",torch.cuda.is_available())

In [ ]:
import ipaddress, csv, urllib.request, itertools
from collections import Counter, defaultdict
import pandas as pd
LABELS = ["NAME", "SURNAME", "INN", "PASSPORT", "CARD", "OGRNIP", "IP", "EMAIL", "BIRTH_CERT", "KLADR"]
SUPPORTED_LABELS = set(LABELS)
PERSON_TYPES = {"NAME", "SURNAME"}
print("Обязательные классы:", LABELS)

## 1. Загрузка обученной RuBERT из GitHub Releases

Архив содержит веса, конфигурацию и токенизатор. Проверяется SHA-256. Google Диск не подключается.


In [ ]:
import shutil
import urllib.error
import urllib.request
import zipfile

GITHUB_REPOSITORY = "SaverPlace/kr22-pdn"
RELEASE_TAG = "v1.0.0"
ARCHIVE_NAME = "rubert_model_v2.zip"
EXPECTED_SHA256 = "0aae9786de78423b50a8611c3b69b3f2e28b8f32cf9af26e0f2ad910bcbd9fca"
EXPECTED_ARCHIVE_SIZE = 658778312
MODEL_DIR = Path("/content/KR22_model/rubert_model_v2")
MODEL_ARCHIVE = Path("/content") / ARCHIVE_NAME


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def model_files_present(directory):
    return all((directory / name).is_file() for name in (
        "config.json", "model.safetensors", "tokenizer_config.json", "vocab.txt"
    )) and any((directory / name).is_file() for name in (
        "tokenizer.json", "vocab.txt"
    ))


def download_model_archive(repository, destination):
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9-]*/[A-Za-z0-9_.-]+", repository):
        raise ValueError("Укажите GITHUB_REPOSITORY в формате пользователь/репозиторий")
    url = f"https://github.com/{repository}/releases/download/{RELEASE_TAG}/{ARCHIVE_NAME}"
    temporary = destination.with_suffix(".zip.part")
    try:
        request = urllib.request.Request(url, headers={"User-Agent": "KR22-Colab"})
        with urllib.request.urlopen(request, timeout=60) as response, temporary.open("wb") as stream:
            shutil.copyfileobj(response, stream, length=1024 * 1024)
        if temporary.stat().st_size != EXPECTED_ARCHIVE_SIZE:
            raise ValueError("Размер скачанного архива не совпадает с исходным")
        if file_sha256(temporary) != EXPECTED_SHA256:
            raise ValueError("Контрольная сумма модели не совпадает с исходным архивом")
        temporary.replace(destination)
    except urllib.error.HTTPError as exc:
        raise RuntimeError(
            f"Не удалось скачать модель (HTTP {exc.code}). Проверьте, что репозиторий "
            f"публичный и в Releases опубликован релиз {RELEASE_TAG} с файлом {ARCHIVE_NAME}."
        ) from exc
    finally:
        temporary.unlink(missing_ok=True)


def unpack_model_archive(archive_path, destination):
    if file_sha256(archive_path) != EXPECTED_SHA256:
        raise ValueError("SHA-256 архива модели не совпадает. Используйте исходный архив из комплекта.")
    destination.parent.mkdir(parents=True, exist_ok=True)
    staging = destination.parent / "unpacking"
    shutil.rmtree(staging, ignore_errors=True)
    staging.mkdir()
    try:
        with zipfile.ZipFile(archive_path) as archive:
            for member in archive.infolist():
                resolved = (staging / member.filename).resolve()
                if not resolved.is_relative_to(staging.resolve()):
                    raise ValueError("Недопустимый путь внутри архива модели")
                if (member.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError("Архив модели содержит символическую ссылку")
            archive.extractall(staging)
        extracted = staging / "rubert_model_v2"
        if not model_files_present(extracted):
            raise FileNotFoundError("В архиве отсутствуют файлы модели или токенизатора")
        shutil.rmtree(destination, ignore_errors=True)
        extracted.replace(destination)
        (destination / ".archive_sha256").write_text(EXPECTED_SHA256, encoding="utf-8")
    finally:
        shutil.rmtree(staging, ignore_errors=True)


marker = MODEL_DIR / ".archive_sha256"
cached = model_files_present(MODEL_DIR) and marker.is_file() and marker.read_text().strip() == EXPECTED_SHA256
if not cached:
    if not MODEL_ARCHIVE.is_file():
        repository = GITHUB_REPOSITORY.strip()
        if not repository:
            repository = input("Введите адрес репозитория в формате пользователь/репозиторий: ").strip()
            repository = repository.removeprefix("https://github.com/").rstrip("/").removesuffix(".git")
        print("Скачивание обученной модели из GitHub Releases...")
        download_model_archive(repository, MODEL_ARCHIVE)
    print("Проверка и распаковка модели...")
    unpack_model_archive(MODEL_ARCHIVE, MODEL_DIR)

person_ner = pipeline(
    task="token-classification",
    model=str(MODEL_DIR),
    tokenizer=str(MODEL_DIR),
    aggregation_strategy="first",
    device=0 if torch.cuda.is_available() else -1,
    model_kwargs={"local_files_only": True},
)
print("Обученная RuBERT загружена:", MODEL_DIR)


## 2. Базовые функции ЛР1 и ЛР2

In [ ]:
def recognize_email(text):
    # Учебный практический шаблон, не исчерпывающий стандарт электронной почты
    pattern = r"(?<![\w@])[A-Za-z0-9._%+-]+@[A-Za-z0-9](?:[A-Za-z0-9.-]*[A-Za-z0-9])?\.[A-Za-z]{2,}(?!\w)"
    return [["EMAIL", m.start(), m.end()] for m in re.finditer(pattern, text)]


def recognize_card(text):
    pattern = r"(?<![\w+])(?:\d{4}(?:[ -]\d{4}){3}|\d{13,19})(?![\w-])"
    return [
        ["CARD", m.start(), m.end()]
        for m in re.finditer(pattern, text)
        if valid_luhn(m.group())
    ]


def recognize_inn(text):
    pattern = r"(?<!\w)(?:\d{12}|\d{10})(?!\w)"
    return [
        ["INN", m.start(), m.end()]
        for m in re.finditer(pattern, text)
        if valid_inn(m.group())
    ]




In [ ]:
import re

# Нейросеть может пропускать короткие анкетные поля: «Имя: Ольга».
# Эти правила применяем только при явной подписи поля.
PERSON_FIELD_RE = {
    "NAME": re.compile(
        r"(?i:\bимя\b(?:\s+(?:сотрудника|клиента|пользователя|человека))?)"
        r"\s*[:—–-]\s*(?P<value>[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?)(?![а-яё-])"
    ),
    "SURNAME": re.compile(
        r"(?i:\bфамилия\b(?:\s+(?:сотрудника|клиента|пользователя|человека))?)"
        r"\s*[:—–-]\s*(?P<value>[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?)(?![а-яё-])"
    ),
    "PATRONYMIC": re.compile(
        r"(?i:\bотчество\b(?:\s+(?:сотрудника|клиента|пользователя|человека))?)"
        r"\s*[:—–-]\s*(?P<value>[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?)(?![а-яё-])"
    ),
}


def recognize_labeled_person(text):
    """ФИО в явно подписанных полях; дополнение к RuBERT, а не замена модели."""
    results = []
    for label, pattern in PERSON_FIELD_RE.items():
        for match in pattern.finditer(text):
            start, end = match.span("value")
            results.append([label, start, end])
    return sorted(results, key=lambda item: item[1])

In [ ]:
ASCII_DIGITS = "0123456789"


def digits_only(value):
    """Список цифр в строке; отклоняет экзотические Unicode-цифры."""
    if not isinstance(value, str):
        raise TypeError("Ожидается строка")
    if any(ch.isdigit() and ch not in ASCII_DIGITS for ch in value):
        raise ValueError("Поддерживаются только цифры 0–9")
    return "".join(ch for ch in value if ch in ASCII_DIGITS)


def restore_separators(template, new_digits):
    """Меняет только позиции цифр, все остальные символы сохраняет дословно."""
    if len(digits_only(template)) != len(new_digits):
        raise ValueError("Число цифр не совпадает с исходным шаблоном")
    iterator = iter(new_digits)
    return "".join(next(iterator) if c in ASCII_DIGITS else c for c in template)


def random_digits(n):
    return "".join(rng.choice(ASCII_DIGITS) for _ in range(n))


def preserve_letter_case(original, new_word):
    if original.isupper():
        return new_word.upper()
    if original.islower():
        return new_word.lower()
    if original.istitle():
        return new_word.capitalize()
    # Для нестандартного чередования регистра не делаем недоказуемых обещаний.
    raise ValueError("Необычный регистр: ожидается имя в нижнем, верхнем или заглавном регистре")

In [ ]:
EMAIL_PATTERN = re.compile(
    r"[A-Za-z0-9._%+-]+@(?:[A-Za-z0-9](?:[A-Za-z0-9-]{0,61}[A-Za-z0-9])?\.)+[A-Za-z]{2,63}"
)


def anonymize_email(value):
    if not isinstance(value, str) or not EMAIL_PATTERN.fullmatch(value):
        raise ValueError(f"Некорректный EMAIL: {value!r}")
    local, domain = value.rsplit("@", 1)
    if local.startswith(".") or local.endswith(".") or ".." in local or len(local) > 64:
        raise ValueError("Некорректная локальная часть e-mail")
    # Минимум 6 знаков локальной части, чтобы обеспечить разнообразие замен.
    size = max(6, len(local))
    alphabet = string.ascii_lowercase + string.digits
    for _ in range(50):
        new_local = "".join(rng.choice(alphabet) for _ in range(size))
        if new_local != local:
            return new_local + "@" + domain
    raise RuntimeError("Не удалось подобрать новую почту")

In [ ]:
INN_WEIGHTS_10 = [2, 4, 10, 3, 5, 9, 4, 6, 8]
INN_WEIGHTS_11 = [7, 2, 4, 10, 3, 5, 9, 4, 6, 8]
INN_WEIGHTS_12 = [3, 7, 2, 4, 10, 3, 5, 9, 4, 6, 8]


def check_digit_inn(prefix, weights):
    return str((sum(int(x) * w for x, w in zip(prefix, weights)) % 11) % 10)


def valid_inn(value):
    if not isinstance(value, str) or not re.fullmatch(r"[0-9]{10}|[0-9]{12}", value):
        return False
    if len(value) == 10:
        return value[9] == check_digit_inn(value[:9], INN_WEIGHTS_10)
    return (
        value[10] == check_digit_inn(value[:10], INN_WEIGHTS_11)
        and value[11] == check_digit_inn(value[:11], INN_WEIGHTS_12)
    )


def anonymize_inn(value):
    if not valid_inn(value):
        raise ValueError(f"Некорректный ИНН / контрольные цифры: {value!r}")
    for _ in range(50):
        if len(value) == 10:
            body = value[:4] + random_digits(5)
            fresh = body + check_digit_inn(body, INN_WEIGHTS_10)
        else:
            body = value[:4] + random_digits(6)
            first = check_digit_inn(body, INN_WEIGHTS_11)
            fresh = body + first + check_digit_inn(body + first, INN_WEIGHTS_12)
        if fresh != value:
            assert valid_inn(fresh)
            return fresh
    raise RuntimeError("Не удалось подобрать новый ИНН")

In [ ]:
CARD_PATTERN = re.compile(r"[0-9]+(?:[ -][0-9]+)*")


def valid_luhn(value):
    if not isinstance(value, str) or not CARD_PATTERN.fullmatch(value):
        return False
    digits = digits_only(value)
    if not (13 <= len(digits) <= 19):
        return False
    total = 0
    for i, ch in enumerate(reversed(digits)):
        digit = int(ch)
        if i % 2 == 1:
            digit *= 2
            if digit > 9:
                digit -= 9
        total += digit
    return total % 10 == 0


def anonymize_card(value):
    if not isinstance(value, str) or not CARD_PATTERN.fullmatch(value):
        raise ValueError(f"Неверный формат CARD: {value!r}")
    digits = digits_only(value)
    if not valid_luhn(value):
        raise ValueError("Номер карты не проходит проверку Луна")
    n = len(digits)
    movable = list(range(6, n - 4))  # сохраняемые первые 6 и последние 4
    if not movable:
        raise ValueError("Недостаточно средних цифр для замены")

    for _ in range(50):
        fresh = list(digits)
        for idx in movable[:-1]:
            fresh[idx] = rng.choice(ASCII_DIGITS)
        # Последнюю изменяемую цифру вычисляем перебором 0..9;
        # это всегда возможно, так как её вклад в сумму Луна — перестановка 0..9.
        options = list(ASCII_DIGITS)
        rng.shuffle(options)
        for test_digit in options:
            fresh[movable[-1]] = test_digit
            candidate = "".join(fresh)
            if candidate != digits and valid_luhn(candidate):
                result = restore_separators(value, candidate)
                assert valid_luhn(result)
                return result
    raise RuntimeError("Не удалось подобрать новый номер карты")

In [ ]:
MALE_NAMES = [
    "Иван", "Алексей", "Дмитрий", "Сергей", "Павел", "Андрей", "Михаил",
    "Николай", "Виктор", "Роман", "Олег", "Константин", "Артём",
    "Максим", "Фёдор", "Глеб", "Пётр", "Игорь", "Евгений", "Антон"
]
FEMALE_NAMES = [
    "Анна", "Мария", "Ольга", "Елена", "Наталья", "Татьяна", "Екатерина",
    "Виктория", "Дарья", "Юлия", "София", "Кристина", "Алёна", "Ирина",
    "Светлана", "Марина", "Алиса", "Галина", "Полина", "Вероника"
]
SURNAME_PAIRS = [
    ("Сидоров", "Сидорова"), ("Петров", "Петрова"),
    ("Волков", "Волкова"), ("Кузнецов", "Кузнецова"),
    ("Белов", "Белова"), ("Смирнов", "Смирнова"),
    ("Орлов", "Орлова"), ("Зайцев", "Зайцева"),
    ("Фёдоров", "Фёдорова"), ("Громов", "Громова"),
    ("Соколов", "Соколова"), ("Жуков", "Жукова"),
    ("Бобров", "Боброва"), ("Романов", "Романова"),
    ("Морозов", "Морозова"), ("Захаров", "Захарова"),
    ("Попов", "Попова"), ("Сергеев", "Сергеева"),
    ("Ковалёв", "Ковалёва"), ("Богданов", "Богданова")
]


def person_feature_matches(parse, field, expected):
    if getattr(parse.tag, field) == expected:
        return True
    if field != "gender" or expected not in ("masc", "femn"):
        return False
    # Словарь может не иметь двух отдельных разборов для имён общего рода.
    return ("Name" in parse.tag and
            (parse.normal_form.lower().replace("ё", "е") in {"саша", "женя"}
             or bool({"ms-f", "Ms-f"} & set(parse.tag.grammemes))))


def analyze_person(value, label, hint=None):
    if label not in ("NAME", "SURNAME"):
        raise ValueError("analyze_person поддерживает только NAME и SURNAME")
    if not isinstance(value, str) or not re.fullmatch(r"[А-Яа-яЁё]+", value):
        raise ValueError("Поддерживаются однословные русские имена и фамилии")
    feature = "Name" if label == "NAME" else "Surn"
    options = [p for p in morph.parse(value) if feature in p.tag]
    hint = hint or {}
    for field in ("gender", "case", "number"):
        if field in hint:
            options = [p for p in options if person_feature_matches(p, field, hint[field])]
    options = [p for p in options if p.tag.number in {"sing", "plur"} and p.tag.case]
    if not options:
        raise ValueError(f"Не удалось установить морфологию {label}: {value!r}; укажите hints")
    return max(options, key=lambda p: p.score)


def _inflect_person_base(base, label, gender, number, gram_case):
    feature = "Name" if label == "NAME" else "Surn"
    possibilities = [
        p for p in morph.parse(base)
        if feature in p.tag and (number == "plur" or p.tag.gender == gender)
    ]
    for parse in possibilities:
        # Род задан лексемой; для словоформы задаём падеж и число.
        candidate = parse.inflect({number, gram_case})
        if candidate is None:
            continue
        if candidate.tag.case != gram_case or candidate.tag.number != number:
            continue
        if number == "sing" and candidate.tag.gender != gender:
            continue
        return candidate.word
    return None


# Резервный анализ отделяем от обычного словарного разбора.
# Никаких вымышленных тегов pymorphy3: результат содержит явный источник признаков.
from dataclasses import dataclass

@dataclass(frozen=True)
class PersonGrammar:
    lemma: str
    gender: str | None
    number: str
    gram_case: str
    source: str


def resolve_person_grammar(value, label, hint=None):
    hint = hint or {}
    if label not in ('NAME', 'SURNAME') or not isinstance(value, str) or not re.fullmatch(r'[А-Яа-яЁё]+', value):
        raise ValueError('Поддерживаются только однословные кириллические имена и фамилии')
    if any(k not in {'gender','number','case'} for k in hint):
        raise ValueError('Неизвестный морфологический признак')
    if ('gender' in hint and hint['gender'] not in ('masc','femn')) or ('number' in hint and hint['number'] not in ('sing','plur')) or ('case' in hint and hint['case'] not in ('nomn','gent','datv','accs','ablt','loct')):
        raise ValueError('Недопустимая морфологическая подсказка')
    try:
        original = analyze_person(value, label, hint)
    except ValueError:
        original = None
    if original is not None:
        gender = hint.get('gender', original.tag.gender)
        number, gram_case = original.tag.number, original.tag.case
        if (number == 'plur' or gender in ('masc','femn')) and gram_case in ('nomn','gent','datv','accs','ablt','loct'):
            return PersonGrammar(original.normal_form, gender, number, gram_case, 'dictionary')

    # Для редкой фамилии метка SURNAME уже пришла от NER / эталона.
    # Позволяем резервный разбор только при полном внешнем согласовании.
    # Для имён без разборов по-прежнему не угадываем грамматику.
    if label == 'SURNAME' and hint.get('number') == 'sing' and hint.get('gender') in ('masc','femn') and hint.get('case') in ('nomn','gent','datv','accs','ablt','loct'):
        parses = [p for p in morph.parse(value) if 'Surn' in p.tag]
        # При известной склоняемой фамилии не отвергаем словарные сведения.
        # Резерв допустим при отсутствии разбора либо неопределённости рода
        # у несклоняемой формы (Fixd). Не переопределяем известный род/падеж.
        compatible = [p for p in parses if p.tag.number in (None,'sing') and p.tag.gender in (None,hint['gender']) and (p.tag.case in (None,hint['case']) or 'Fixd' in p.tag)]
        if not parses or compatible:
            lemma = max(compatible,key=lambda p:p.score).normal_form if compatible else value.lower()
            return PersonGrammar(lemma, hint['gender'], 'sing', hint['case'], 'context_fallback')
    raise ValueError(f'Недостаточно надёжных грамматических признаков {label}: {value!r}; укажите hints')


def anonymize_person(value, label, *, hint=None, memo=None, audit=None):
    original = resolve_person_grammar(value, label, hint)
    gender, number, gram_case = original.gender, original.number, original.gram_case
    if label == 'NAME':
        choices = (MALE_NAMES if gender == 'masc' else FEMALE_NAMES) if number == 'sing' else MALE_NAMES + FEMALE_NAMES
    else:
        choices = ([m for m, f in SURNAME_PAIRS] if gender == 'masc' else [f for m, f in SURNAME_PAIRS]) if number == 'sing' else [m for m, f in SURNAME_PAIRS]
    pool = choices[:]
    rng.shuffle(pool)
    # Без словарной леммы ключ содержит исходную словоформу. Разные формы
    # неизвестной склоняемой фамилии не объединяем без доказательств.
    memo_key = (label, original.lemma, gender, number)
    if memo is not None and memo_key in memo:
        pool = [memo[memo_key]]
    for base in pool:
        if base.lower() == original.lemma.lower():
            continue
        result = _inflect_person_base(base, label, gender, number, gram_case)
        if result is None or result.lower() == value.lower():
            continue
        replacement = preserve_letter_case(value, result)
        if memo is not None and memo_key not in memo:
            memo[memo_key] = base
        if audit is not None and original.source == 'context_fallback':
            audit.append({'label':label,'original':value,'gender':gender,'number':number,'case':gram_case,'replacement':replacement,'source':original.source})
        return replacement
    raise ValueError(f'Не удалось подобрать форму для {value!r} ({label}, {gram_case})')


## 3. Справочник КЛАДР

Источники: https://github.com/afa/ruadm ; https://base.garant.ru/12144391/53f89421bbdaf741eb2d1ecc4ddb4c33/ (структура КЛАДР).

In [ ]:
KLADR_CSV = Path('/content/kladr.csv')
KLADR_URL = 'https://raw.githubusercontent.com/afa/ruadm/master/data/kladr.csv'
if not KLADR_CSV.exists():
    try:
        urllib.request.urlretrieve(KLADR_URL, str(KLADR_CSV))
    except Exception as exc:
        raise RuntimeError('Загрузите kladr.csv вручную в /content или укажите свой путь') from exc

def load_kladr(path):
    by_code, groups = {}, defaultdict(list)
    with open(path, encoding='utf-8-sig', newline='') as f:
        for row in csv.DictReader(f):
            c = row['code'].strip()
            level = int(row['level'])
            if not re.fullmatch(r'(?:[0-9]{13}|[0-9]{17})', c):
                continue
            if c in by_code:
                continue
            key = (len(c), c[:2], level, c[-2:])
            by_code[c] = key
            groups[key].append(c)
    if not by_code:
        raise ValueError('Пустой справочник КЛАДР')
    return by_code, groups

KLADR_INDEX, KLADR_GROUPS = load_kladr(KLADR_CSV)
print('Кодов в справочнике:', len(KLADR_INDEX))
print('SHA256 справочника:',hashlib.sha256(KLADR_CSV.read_bytes()).hexdigest())

## 4. Новые атрибуты: проверка и распознавание
Паспорт РФ: 4 цифры серии + 6 цифр номера, слитные 10 цифр только при подсказке «паспорт». Свидетельство о рождении: римская часть + 2 кириллические буквы + 6 цифр; для обеих проверяется формат, не выдача документа. ОГРНИП: 15 цифр, первая 3, контрольная `(int(первые 14) % 13) % 10`. IPv4/IPv6 проверяются стандартным модулем `ipaddress`. КЛАДР определяется по контексту и справочнику, поскольку длина пересекается с банковскими картами. ИНН и карты: сохранены математические проверки ЛР1/2. Приоритет контекстно подписанных полей предотвращает подмену ошибочного ОГРНИП картой или ошибочного паспорта ИНН.

Источники форматов: https://www.consultant.ru/document/cons_doc_LAW_288080/bfda7a84013d0ad8a3d6e57b229b1bbcbf923868/ .

Ограничение: модель RuBERT в этой версии рассчитана на короткие тексты в пределах её контекстного окна; длинные документы следует предварительно разделять на фрагменты с пересчётом смещений. Необнаруженная сущность не будет обезличена.

In [ ]:
PASSPORT_PATTERN = re.compile(r'(?:[0-9]{2}[ -]?[0-9]{2}[ \t-]*(?:№[ \t]*)?[0-9]{6})')
BIRTH_PATTERN = re.compile(r'(?i:[IVX]{1,5})[ -]?[А-ЯЁа-яё]{2}[ \t-]*(?:№[ \t]*)?[0-9]{6}')
FIELD_HINTS = {
    'PASSPORT': r'паспорт\w*|серия\s+и\s+номер\s+паспорта',
    'INN': r'инн', 'OGRNIP': r'огрнип', 'KLADR': r'(?:код\s+)?кладр',
    'CARD': r'(?:банковск\w+\s+)?карт[аы]'}

def field_hint(text, start):
    before = text[max(0,start-65):start]
    for label, pattern in FIELD_HINTS.items():
        if re.search(r'(?i)\b(?:'+pattern+r')\s*(?:[:=№—-]\s*)?$', before):
            return label
    return None

def valid_email(v):
    if not isinstance(v,str) or not EMAIL_PATTERN.fullmatch(v): return False
    local=v.rsplit('@',1)[0]
    return 0<len(local)<=64 and not local.startswith('.') and not local.endswith('.') and '..' not in local

def valid_passport(v):
    return isinstance(v,str) and bool(PASSPORT_PATTERN.fullmatch(v)) and len(digits_only(v))==10

def valid_birth_certificate(v):
    return isinstance(v,str) and bool(BIRTH_PATTERN.fullmatch(v))

def valid_ogrnip(v):
    return isinstance(v,str) and bool(re.fullmatch(r'3[0-9]{14}',v)) and int(v[-1]) == int(v[:-1]) % 13 % 10

def valid_kladr(v):
    return isinstance(v,str) and v in KLADR_INDEX

def valid_ip(v):
    try:
        if not isinstance(v,str) or '%' in v: return False
        ipaddress.ip_address(v)
        return True
    except ValueError: return False

def scan_pattern(text,label,pattern,validator,allow=lambda t,m: True):
    return [[label,m.start(),m.end()] for m in re.finditer(pattern,text)
            if validator(m.group()) and allow(text,m)]

def recognize_passport(text):
    return scan_pattern(text,'PASSPORT',r'(?<![\w])'+PASSPORT_PATTERN.pattern+r'(?![\w])',valid_passport,
          lambda t,m: field_hint(t,m.start())=='PASSPORT' or (field_hint(t,m.start()) is None and not m.group().isdigit()))

def recognize_ogrnip(text):
    return scan_pattern(text,'OGRNIP',r'(?<!\w)3[0-9]{14}(?!\w)',valid_ogrnip,
                        lambda t,m: field_hint(t,m.start()) in (None,'OGRNIP'))

def recognize_birth_certificate(text):
    return scan_pattern(text,'BIRTH_CERT',r'(?<!\w)'+BIRTH_PATTERN.pattern+r'(?!\w)',valid_birth_certificate)

def recognize_kladr(text):
    return scan_pattern(text,'KLADR',r'(?<!\w)(?:[0-9]{17}|[0-9]{13})(?!\w)',valid_kladr,
                        lambda t,m: field_hint(t,m.start()) in (None,'KLADR'))

def recognize_ip(text):
    result = scan_pattern(text,'IP',r'(?<![\w.])(?:[0-9]{1,3}\.){3}[0-9]{1,3}(?!\w|\.\d)',valid_ip)
    for m in re.finditer(r'(?<![\w:])(?:[0-9A-Fa-f:.]*:[0-9A-Fa-f:.]+|::)(?![\w:])',text):
        v=m.group().rstrip('.')
        if ':' in v and valid_ip(v): result.append(['IP',m.start(),m.start()+len(v)])
    return result

def resolve_overlaps(text,candidates):
    priority={'EMAIL':0,'IP':1,'BIRTH_CERT':2,'PASSPORT':3,'OGRNIP':4,'KLADR':5,'INN':6,'CARD':7,'NAME':8,'SURNAME':8}
    def rank(x):
        lab,s,e=x
        explicit = field_hint(text,s)==lab
        return (0 if explicit else 1,priority[lab],-(e-s),s,lab)
    out=[]
    for lab,s,e in sorted(candidates,key=rank):
        if not any(s<b and a<e for _,a,b in out): out.append([lab,s,e])
    return sorted(out,key=lambda x:x[1])

def recognize_pdn(text,raw_ner=None):
    if not isinstance(text,str): raise TypeError('Ожидается текст')
    structured = ([x for x in recognize_email(text) if valid_email(text[x[1]:x[2]])]+recognize_ip(text)+recognize_birth_certificate(text)+
                  recognize_passport(text)+recognize_ogrnip(text)+recognize_kladr(text)+
                  recognize_inn(text)+recognize_card(text))
    structured=[x for x in structured if field_hint(text,x[1]) in (None,x[0])]
    labeled=[x for x in recognize_labeled_person(text) if x[0] in PERSON_TYPES]
    raw=person_ner(text) if raw_ner is None else raw_ner
    ner=[[i['entity_group'].upper(),int(i['start']),int(i['end'])] for i in raw
         if i.get('entity_group','').upper() in PERSON_TYPES]
    ner=[x for x in ner if not any(x[1]<b and a<x[2] for _,a,b in labeled)]
    return resolve_overlaps(text,structured+labeled+ner)

## 5. Обезличивание новых классов


In [ ]:
def random_other_digits(old):
    for _ in range(100):
        candidate=random_digits(len(old))
        if candidate!=old: return candidate
    raise RuntimeError('Не удалось сгенерировать отличие')

def anonymize_passport(v):
    if not valid_passport(v): raise ValueError('Неверный формат паспорта')
    d=digits_only(v)
    return restore_separators(v,d[:4]+random_other_digits(d[4:]))

def anonymize_birth_certificate(v):
    if not valid_birth_certificate(v): raise ValueError('Неверный формат свидетельства')
    return v[:-6]+random_other_digits(v[-6:])

def anonymize_ogrnip(v):
    if not valid_ogrnip(v): raise ValueError('Неверный ОГРНИП')
    for _ in range(100):
        body=v[:5]+random_digits(9)
        new=body+str(int(body)%13%10)
        if new!=v: return new
    raise RuntimeError('Не удалось сгенерировать ОГРНИП')

def ip_kind(a):
    return (a.is_private,a.is_global,a.is_multicast,a.is_loopback,a.is_link_local,a.is_reserved,a.is_unspecified)

def anonymize_ip(v):
    if not valid_ip(v): raise ValueError('Неверный IP')
    a=ipaddress.ip_address(v)
    bits=8 if a.version==4 else 64
    prefix=int(a)>>bits<<bits
    for _ in range(500):
        tail=rng.randint(1,254) if a.version==4 else rng.getrandbits(bits)
        b=type(a)(prefix|tail)
        if b!=a and ip_kind(a)==ip_kind(b): return str(b)
    raise ValueError('В данном префиксе не найден допустимый IP другого значения')

def anonymize_kladr(v):
    if not valid_kladr(v): raise ValueError('Код отсутствует в справочнике')
    options=KLADR_GROUPS[KLADR_INDEX[v]]
    if len(options)<2: raise ValueError('В группе КЛАДР нет альтернативы')
    while True:
        result=rng.choice(options)
        if result!=v: return result

## 6. Контекстная морфология — устранение неоднозначности и редкие фамилии


In [ ]:
def person_context_hint(text,label,start,end,entities=None):
    """Узкие правила согласования, не полноценный синтаксический анализ."""
    word=text[start:end]
    feature='Name' if label=='NAME' else 'Surn'
    parses=[p for p in morph.parse(word) if feature in p.tag]
    def compatible(h):
        # Не блокируем внешнюю подсказку из-за отсутствия редкой фамилии в словаре.
        if label=='SURNAME' and not parses:
            return True
        for p in parses:
            if all(person_feature_matches(p,k,v) or
                   (label=='SURNAME' and getattr(p.tag,k) is None) or
                   (label=='SURNAME' and k=='case' and 'Fixd' in p.tag)
                   for k,v in h.items()):
                return True
        return False

    # 1. Ближайший глагол прошедшего времени, только при непосредственном
    # соседстве. Для известного слова не допускаем конфликт со словарём.
    after=re.match(r'\s+([А-Яа-яЁё]+)',text[end:])
    if after:
        for p in morph.parse(after[1]):
            if p.tag.POS=='VERB' and p.tag.tense=='past' and p.tag.number=='sing' and p.tag.gender in ('masc','femn'):
                h={'gender':p.tag.gender,'number':'sing','case':'nomn'}
                if compatible(h):
                    return h

    # 2. Согласование фамилии с соседним именем, уже выделенным NER.
    # Не берём случайное слово из окружающего текста за имя человека.
    neighbor=None
    if label=='SURNAME' and entities is not None:
        for lab,s,e in entities:
            if lab!='NAME': continue
            if e<=start and re.fullmatch(r'[ \t]+',text[e:start]):
                neighbor=(s,e); break
        if neighbor is None:
            for lab,s,e in entities:
                if lab=='NAME' and s>=end and re.fullmatch(r'[ \t]+',text[end:s]):
                    neighbor=(s,e); break
    cases={'к':{'datv'},'ко':{'datv'},'без':{'gent'},'для':{'gent'},'от':{'gent'},
           'у':{'gent'},'о':{'loct'},'об':{'loct'},'над':{'ablt'},'перед':{'ablt'},'с':{'ablt','gent'}}
    phrase_start=min(start,neighbor[0]) if neighbor else start
    before=re.search(r'([А-Яа-яЁё]+)\s*$',text[:phrase_start])
    allowed=cases.get(before[1].lower(),set()) if before else set()
    if neighbor:
        s,e=neighbor
        # Словарные варианты имени. Не используем max(score) как доказательство рода.
        options=set()
        for p in morph.parse(text[s:e]):
            if 'Name' not in p.tag or p.tag.number!='sing' or p.tag.gender not in ('masc','femn') or p.tag.case not in ('nomn','gent','datv','accs','ablt','loct'):
                continue
            if p.normal_form.lower().replace('ё','е') in {'саша','женя'} or {'ms-f','Ms-f'} & set(p.tag.grammemes):
                continue
            if allowed and p.tag.case not in allowed: continue
            h={'gender':p.tag.gender,'number':'sing','case':p.tag.case}
            if compatible(h): options.add((p.tag.gender,p.tag.case))
        if len(options)==1:
            gender,gram_case=next(iter(options))
            return {'gender':gender,'number':'sing','case':gram_case}
        # Если есть несколько совместимых вариантов, сохраняем только общие
        # признаки. Нельзя выдумывать падеж редкой фамилии.
        if options:
            genders={g for g,c in options}; gram_cases={c for g,c in options}
            hint={'number':'sing'}
            if len(genders)==1: hint['gender']=next(iter(genders))
            if len(gram_cases)==1: hint['case']=next(iter(gram_cases))
            return hint

    # 3. Предлог непосредственно перед сущностью (или парой имя + фамилия).
    possible={p.tag.case for p in parses if p.tag.case in allowed}
    return {'case':next(iter(possible))} if len(possible)==1 else {}


## 7. Единая программа и согласованность замен


In [ ]:
_LAST_RESULT_DIGEST = {}  # только хеши; исходные ПДн здесь не кэшируются


def normalize_entities(text, entities):
    if not isinstance(text, str):
        raise TypeError("text должен быть строкой")
    if isinstance(entities, (list, tuple)) and len(entities) == 3 and isinstance(entities[0], str):
        entities = [entities]
    if not isinstance(entities, (list, tuple)):
        raise TypeError("entities должен быть списком троек [тип, start, end]")
    normalized = []
    for item in entities:
        if not isinstance(item, (tuple, list)) or len(item) != 3:
            raise ValueError(f"Ожидается тройка [тип, start, end]: {item!r}")
        label, start, end = item
        if not isinstance(label, str) or label not in SUPPORTED_LABELS:
            raise ValueError(f"Тип {label!r} не поддержан вариантом 22; разрешены {sorted(SUPPORTED_LABELS)}")
        if type(start) is not int or type(end) is not int or not (0 <= start < end <= len(text)):
            raise ValueError(f"Некорректные границы сущности: {item!r}")
        normalized.append((label, start, end))
    normalized.sort(key=lambda row: (row[1], row[2]))
    for a, b in zip(normalized, normalized[1:]):
        if a[2] > b[1]:
            raise ValueError(f"Перекрывающиеся сущности: {a!r} и {b!r}")
    return normalized

In [ ]:
HANDLERS={'EMAIL':anonymize_email,'INN':anonymize_inn,'CARD':anonymize_card,
          'PASSPORT':anonymize_passport,'OGRNIP':anonymize_ogrnip,'IP':anonymize_ip,
          'BIRTH_CERT':anonymize_birth_certificate,'KLADR':anonymize_kladr}
VALIDATORS={'EMAIL':valid_email, 'INN':valid_inn,
            'CARD':lambda v: bool(CARD_PATTERN.fullmatch(v)) and valid_luhn(v),
            'PASSPORT':valid_passport,'OGRNIP':valid_ogrnip,'IP':valid_ip,
            'BIRTH_CERT':valid_birth_certificate,'KLADR':valid_kladr}

def replace_one(value,label,*,hint=None,memo=None,audit=None):
    if memo is None: memo={}
    if label in PERSON_TYPES: return anonymize_person(value,label,hint=hint,memo=memo,audit=audit)
    if not VALIDATORS[label](value): raise ValueError('Неверный формат или контрольное число '+label)
    if label in {'CARD','PASSPORT'}: canonical=digits_only(value)
    elif label=='IP': canonical=str(ipaddress.ip_address(value))
    elif label=='BIRTH_CERT': canonical=re.sub(r'[\s№-]','',value).upper()
    else: canonical=value
    key=(label,canonical)
    if key not in memo:
        replacement=HANDLERS[label](value)
        if label in {'CARD','PASSPORT'}: replacement=digits_only(replacement)
        elif label=='BIRTH_CERT': replacement=replacement[-6:]
        memo[key]=replacement
    replacement=memo[key]
    if label in {'CARD','PASSPORT'}: return restore_separators(value,replacement)
    if label=='BIRTH_CERT': return value[:-6]+replacement
    return replacement

def anonymize(text,entities,*,hints=None,on_error='raise',memo=None,errors=None,audit=None):
    items=normalize_entities(text,entities)
    if on_error not in {'raise','redact'}: raise ValueError('Неверный on_error')
    if memo is None: memo={}
    hints=hints or {}
    parts=[]; cursor=0
    for label,start,end in items:
        parts.append(text[cursor:start])
        hint=person_context_hint(text,label,start,end,items) if label in PERSON_TYPES else {}
        if (start,end) in hints: hint.update(hints[(start,end)])
        try:
            replacement=replace_one(text[start:end],label,hint=hint,memo=memo,audit=audit)
        except (ValueError,TypeError,RuntimeError) as exc:
            if on_error=='raise': raise
            if errors is not None: errors.append({'label':label,'start':start,'end':end,'error':str(exc)})
            replacement=f'[{label}]'
        parts.append(replacement); cursor=end
    parts.append(text[cursor:])
    return ''.join(parts)

def anon(text,entities=None,*,hints=None,on_error='raise'):
    items=recognize_pdn(text) if entities is None else normalize_entities(text,entities)
    items=[list(x) for x in items]
    signature=hashlib.sha256(repr((text,items,sorted((hints or {}).items()),on_error)).encode()).digest()
    for _ in range(40):
        errors=[]
        result=anonymize(text,items,hints=hints,on_error=on_error,errors=errors)
        digest=hashlib.sha256(result.encode()).digest()
        # Если нет замен или все замены — маркеры, разнообразие невозможно.
        if not items or len(errors)==len(items) or digest!=_LAST_RESULT_DIGEST.get(signature):
            if len(_LAST_RESULT_DIGEST)>=1000: _LAST_RESULT_DIGEST.clear()
            _LAST_RESULT_DIGEST[signature]=digest
            return result,items
    raise RuntimeError('Не удалось получить результат, отличный от предыдущего')

def mask(text,entities):
    parts=[]; cursor=0
    for _,s,e in normalize_entities(text,entities):
        parts.extend([text[cursor:s],'*'*(e-s)]); cursor=e
    parts.append(text[cursor:]); return ''.join(parts)

## 8. Контрольные проверки до итоговой оценки
Проверяем правила сохранения структуры, морфологию, исключения, согласованные замены и изменение результата при повторном вызове. Это диагностические тесты, не включаем их в итоговый F1.

In [ ]:
def make_inn12(n):
    body=f'7706{n:06d}'
    body+=check_digit_inn(body,INN_WEIGHTS_11)
    return body+check_digit_inn(body,INN_WEIGHTS_12)

def make_inn10(n):
    body=f'5001{n:05d}'
    return body+check_digit_inn(body,INN_WEIGHTS_10)

def make_card(n):
    body=f'400000{n:09d}'
    return next(body+str(d) for d in range(10) if valid_luhn(body+str(d)))

def make_ogrnip(n):
    body=f'32477{n:09d}'
    return body+str(int(body)%13%10)

def check_structured(label,old,new):
    assert old!=new and VALIDATORS[label](new), (label,old,new)
    if label=='EMAIL': assert old.rsplit('@',1)[1]==new.rsplit('@',1)[1]
    elif label=='INN': assert len(old)==len(new) and old[:4]==new[:4]
    elif label=='CARD':
        a,b=digits_only(old),digits_only(new)
        assert len(a)==len(b) and a[:6]==b[:6] and a[-4:]==b[-4:]
        assert len(old)==len(new) and all(x==y for x,y in zip(old,new) if x not in ASCII_DIGITS)
    elif label=='PASSPORT':
        assert digits_only(old)[:4]==digits_only(new)[:4]
        assert len(old)==len(new) and all(x==y for x,y in zip(old,new) if x not in ASCII_DIGITS)
    elif label=='BIRTH_CERT': assert old[:-6]==new[:-6]
    elif label=='OGRNIP': assert old[:5]==new[:5]
    elif label=='IP':
        a,b=ipaddress.ip_address(old),ipaddress.ip_address(new)
        bits=24 if a.version==4 else 64
        assert a!=b and b in ipaddress.ip_network(f'{a}/{bits}',strict=False) and ip_kind(a)==ip_kind(b)
    elif label=='KLADR': assert KLADR_INDEX[old]==KLADR_INDEX[new]

eligible_kladr=sorted(c for c,k in KLADR_INDEX.items() if len(KLADR_GROUPS[k])>=2 and len(c)==13)
assert len(eligible_kladr)>=50,'Нужны минимум 50 кодов с доступными заменами'
examples={'EMAIL':'demo@example.org','INN':make_inn12(321),'CARD':'4561 2612 1234 5467',
          'PASSPORT':'45 08 № 123456','OGRNIP':make_ogrnip(1234),'IP':'192.168.7.35',
          'BIRTH_CERT':'II-АБ № 123456','KLADR':eligible_kladr[0]}
checks=0
for label,v in examples.items():
    for _ in range(20):
        check_structured(label,v,HANDLERS[label](v)); checks+=1
for v in ['2001:db8:12:34::1234','10.12.23.34']:
    check_structured('IP',v,anonymize_ip(v)); checks+=1
for label,v in [('OGRNIP',make_ogrnip(11)[:-1]+str((int(make_ogrnip(11)[-1])+1)%10)),
                ('IP','256.10.10.10'),('PASSPORT','123'),('BIRTH_CERT','X-AB 123456'),
                ('KLADR','0000000000000'),('INN','7707083894'),('CARD','1234'),('EMAIL','no-at-sign')]:
    try: HANDLERS[label](v)
    except (ValueError,TypeError): checks+=1
    else: raise AssertionError('Не отклонено: '+label)
# Проверка обеих разновидностей неоднозначного имени.
for text,gender in [('Саша пришёл домой.','masc'),('Саша пришла домой.','femn')]:
    hint=person_context_hint(text,'NAME',0,4)
    assert hint.get('gender')==gender, (text,hint)
    replacement=anonymize_person('Саша','NAME',hint=hint)
    p=analyze_person(replacement,'NAME',hint)
    assert p.tag.gender==gender and p.tag.case=='nomn'
    print(text,'→',replacement+text[4:]); checks+=1
# Морфология: проверяем по заранее заданным грамматическим признакам, а не по тому же max(score).
for label,v,hint in [
    ('NAME','Ивану',{'gender':'masc','number':'sing','case':'datv'}),
    ('NAME','Марией',{'gender':'femn','number':'sing','case':'ablt'}),
    ('SURNAME','Петрова',{'gender':'femn','number':'sing','case':'nomn'}),
    ('SURNAME','Петрова',{'gender':'masc','number':'sing','case':'accs'}),
    ('SURNAME','Сидоровыми',{'number':'plur','case':'ablt'})]:
    for _ in range(8):
        replacement=anonymize_person(v,label,hint=hint)
        analyze_person(replacement,label,hint)
        assert replacement.lower()!=v.lower()
        checks+=1
text='Иван пришёл. Иван ушёл. Почта: x@example.org; E-mail: x@example.org.'
e=[]
for label,value in [('NAME','Иван'),('EMAIL','x@example.org')]:
    e.extend([label,m.start(),m.end()] for m in re.finditer(re.escape(value),text))
new,_=anon(text,e); new2,_=anon(text,e)
assert new!=new2
assert re.findall(r'[\w.]+@example\.org',new)[0]==re.findall(r'[\w.]+@example\.org',new)[1]
assert new.split()[0]==new.split('.')[1].split()[0]
for label,value in examples.items():
    t=value+'; '+value
    ents=[[label,0,len(value)],[label,len(value)+2,len(t)]]
    a,b=anonymize(t,ents).split('; ')
    assert a==b
print('Структурные проверки, ошибки и морфология:',checks,'; согласованность и повторные вызовы: OK')
# Одна и та же лемма в разных падежах в пределах документа.
m={}
a=anonymize_person('Иван','NAME',hint={'gender':'masc','case':'nomn'},memo=m)
b=anonymize_person('Ивану','NAME',hint={'gender':'masc','case':'datv'},memo=m)
assert analyze_person(a,'NAME',{'gender':'masc','case':'nomn'}).normal_form==analyze_person(b,'NAME',{'gender':'masc','case':'datv'}).normal_form
print('Согласованная морфология:',a,b)

In [ ]:
# Дополнительные регрессионные тесты морфологии (не входят в итоговый датасет).
# Ожидаемые грамматические признаки задаются вручную независимо от алгоритма.
regression=[
    ('Анна Петрова подписала договор.','Анна','Петрова','femn','nomn'),
    ('Я передал документы Алексею Чжоу.','Алексею','Чжоу','masc','datv'),
    ('Мы поговорили с Марией Ким.','Марией','Ким','femn','ablt'),
    ('Иван Зорбашевич подготовил отчёт.','Иван','Зорбашевич','masc','nomn'),
    ('Алексей Нурбазаде приехал утром.','Алексей','Нурбазаде','masc','nomn'),
    ('Документы доставили к Анне Лушенко.','Анне','Лушенко','femn','datv'),
]
regression_results=[]
for text,name,surname,gender,gram_case in regression:
    ns=text.index(name); ss=text.index(surname)
    ents=[['NAME',ns,ns+len(name)],['SURNAME',ss,ss+len(surname)]]
    hint=person_context_hint(text,'SURNAME',ss,ss+len(surname),ents)
    expected={'gender':gender,'number':'sing','case':gram_case}
    try:
        assert all(hint.get(k)==v for k,v in expected.items()), (hint,expected)
        grammar=resolve_person_grammar(surname,'SURNAME',hint)
        assert (grammar.gender,grammar.number,grammar.gram_case)==(gender,'sing',gram_case)
        memo={}; audit=[]
        replacement=anonymize_person(surname,'SURNAME',hint=hint,memo=memo,audit=audit)
        analyze_person(replacement,'SURNAME',expected)
        assert replacement.casefold()!=surname.casefold()
        assert anonymize_person(surname,'SURNAME',hint=hint,memo=memo)==replacement
        regression_results.append({'text':text,'hint':hint,'source':grammar.source,'replacement':replacement,'ok':True})
    except (AssertionError,ValueError,RuntimeError) as exc:
        regression_results.append({'text':text,'hint':hint,'ok':False,'error':str(exc)})
# Гарантия, что неверная / неполная подсказка не приводит к угадыванию.
try: resolve_person_grammar('Зорбашевич','SURNAME',{'gender':'unknown'})
except ValueError: print('Некорректная подсказка отклонена: OK')
else: raise AssertionError('Некорректная подсказка принята')
display(pd.DataFrame(regression_results))
print('Новые проверки морфологии:',sum(x['ok'] for x in regression_results),'/',len(regression_results))
# Не останавливаем эксперимент BERTScore, даже если на новом наборе найдены ограничения.


## 9. Независимая тестовая выборка


In [ ]:
# Только генератор тестовой выборки фиксирован; генератор обезличивания остается SystemRandom.
test_rng=random.Random(22026)

def labeled_case(parts,source='synthetic'):
    text=''; spans=[]
    for item in parts:
        if isinstance(item,tuple):
            label,v=item; s=len(text); text+=v; spans.append([label,s,len(text)])
        else: text+=item
    return {'text':text,'entities':spans,'source':source}

# Формы составлены заранее; шаблоны грамматически согласованы.
name_forms=[('Кирилл','Кириллу','Кириллом'),('Лев','Льву','Львом'),('Борис','Борису','Борисом'),
 ('Матвей','Матвею','Матвеем'),('Григорий','Григорию','Григорием'),('Аркадий','Аркадию','Аркадием'),
 ('Станислав','Станиславу','Станиславом'),('Ярослав','Ярославу','Ярославом'),
 ('Алиса','Алисе','Алисой'),('Вера','Вере','Верой'),('Лидия','Лидии','Лидией'),
 ('Элина','Элине','Элиной'),('Лариса','Ларисе','Ларисой'),('Нина','Нине','Ниной'),
 ('Зоя','Зое','Зоей'),('Регина','Регине','Региной'),('Кристина','Кристине','Кристиной'),
 ('Виктория','Виктории','Викторией'),('Юлия','Юлии','Юлией'),('Раиса','Раисе','Раисой')]
surname_forms=[('Калинин','Калинину','Калининым'),('Тихонов','Тихонову','Тихоновым'),('Комаров','Комарову','Комаровым'),
 ('Макаров','Макарову','Макаровым'),('Денисов','Денисову','Денисовым'),('Киселёв','Киселёву','Киселёвым'),
 ('Крылов','Крылову','Крыловым'),('Никитин','Никитину','Никитиным'),('Лебедева','Лебедевой','Лебедевой'),
 ('Власова','Власовой','Власовой'),('Котова','Котовой','Котовой'),('Гусева','Гусевой','Гусевой'),
 ('Андреева','Андреевой','Андреевой'),('Яковлева','Яковлевой','Яковлевой'),('Степанова','Степановой','Степановой'),
 ('Кудрявцева','Кудрявцевой','Кудрявцевой'),('Быкова','Быковой','Быковой')]

def person_template(label,forms,i):
    idx=i%len(forms); form=(i//len(forms))%3
    v=forms[idx][form]
    if form==0:
        verb='подтвердила' if (idx>=8) else 'подтвердил'
        if i%2: return [(label,v),f' {verb} получение документов.']
        return [('Имя: ' if label=='NAME' else 'Фамилия: '),(label,v),'. Заявление принято.']
    if form==1: return ['Копию заявления направили ',(label,v),'. Ответ ожидается на неделе.']
    return ['На встрече обсудили договор с ',(label,v),'.']

kladr_values=test_rng.sample(eligible_kladr,50)

def structured_example(label,i):
    if label=='EMAIL': return f'contact.{230+i}+data@'+['sample.example.org','demo.example.net','example.com'][i%3]
    if label=='INN': return make_inn10(72000+i) if i%2 else make_inn12(430000+i)
    if label=='PASSPORT':
        d=f'52{8+i%15:02d}{345800+i:06d}'
        return [d,d[:2]+' '+d[2:4]+' '+d[4:],d[:4]+' № '+d[4:]][i%3]
    if label=='CARD':
        d=make_card(835472+i*31)
        return d if i%3==0 else (' ' if i%3==1 else '-').join(d[j:j+4] for j in range(0,16,4))
    if label=='OGRNIP': return make_ogrnip(527134+i*19)
    if label=='IP': return f'192.168.{10+i%8}.{20+i}' if i%3 else f'2001:db8:{i+11:x}:42::{i+3:x}'
    if label=='BIRTH_CERT': return ['IV-АЖ № ','II-ВК ','III-РТ'][i%3]+f'{682450+i:06d}'
    if label=='KLADR': return kladr_values[i%50]

field_names={'EMAIL':'Электронная почта','INN':'ИНН','PASSPORT':'Паспорт','CARD':'Карта',
             'OGRNIP':'ОГРНИП','IP':'IP-адрес','BIRTH_CERT':'Свидетельство о рождении','KLADR':'Код КЛАДР'}

def structured_template(label,i):
    value=structured_example(label,i)
    field=field_names[label]
    # Паспорт без разделителей требует явной подписи; при прочих форматах тоже варьируем ввод.
    if i%3==0: return [field+': ',(label,value),'. Проверка завершена.']
    if i%3==1: return ['В карточке указан '+field+' — ',(label,value),'; запись принята.']
    return ['Реквизиты для договора. '+field+' ',(label,value),'.']

cases=[]
for lab in LABELS:
    for i in range(50):
        parts=person_template(lab,name_forms if lab=='NAME' else surname_forms,i) if lab in PERSON_TYPES else structured_template(lab,i)
        cases.append(labeled_case(parts))
for i in range(25):
    a,b=test_rng.sample(list(field_names),2)
    cases.append(labeled_case(['В заявлении приведены сведения. ',field_names[a]+': ',(a,structured_example(a,i)),
                               '; '+field_names[b]+': ',(b,structured_example(b,i+25)),'.']))
negatives=['На совещании обсуждали бюджет и сроки.','В аудитории установили новый компьютер.',
 'Рабочая группа согласовала проект.','Отчёт подготовлен в электронном виде.',
 'Неверный IP-адрес: 256.10.10.10.','ОГРНИП: 123456789012345.','ИНН: 1234567890.',
 'Электронная почта отсутствует.','Паспорт: 1234.','Карта: 1234567890123456.',
 'Код КЛАДР: 0000000000000.','Свидетельство о рождении: 123.',
 'Устройство выключено.','Документ отправлен на проверку.','Сегодня состоялось собрание.',
 'Письмо доставлено по назначению.','Склад работает в штатном режиме.',
 'Необходимо заполнить заявление.','Результаты внесены в таблицу.','Реквизиты не указаны.',
 'Проверка формата завершена.','Запрос отклонён.','Регистрация пока недоступна.',
 'Отправка запланирована на утро.','Сотрудники обсуждали новый порядок работы.']
cases.extend({'text':t,'entities':[],'source':'synthetic_negative'} for t in negatives)

In [ ]:
corpus_cases = [{'text': 'Жизнь с самого начала оставляла для Александра Вертинского слишком много вопросов без ответов.', 'entities': [['NAME', 36, 46], ['SURNAME', 47, 58]], 'source': 'factRuEval2016/testset/book_3539', 'source_start': 194}, {'text': 'Дмитрий Медведев доверил пост руководителя иннограда миллиардеру Виктору Вексельбергу.', 'entities': [['NAME', 0, 7], ['SURNAME', 8, 16], ['NAME', 65, 72], ['SURNAME', 73, 85]], 'source': 'factRuEval2016/testset/book_3543', 'source_start': 61}, {'text': 'В середине девяностых в Японии российский переводчик Дмитрий Коваленин попросил знакомого бармена порекомендовать ему что-нибудь из местной литературы, чтобы «было интересно молодёжи во всём в мире, а не только японской».', 'entities': [['NAME', 53, 60], ['SURNAME', 61, 70]], 'source': 'factRuEval2016/testset/book_3555', 'source_start': 1390}, {'text': 'Эти обстоятельства подрывают веру местного населения и всего мирового сообщества в законность и благонадёжность правительства Хамида Карзая и усиливают авторитет боевиков.', 'entities': [['NAME', 126, 132], ['SURNAME', 133, 139]], 'source': 'factRuEval2016/testset/book_3562', 'source_start': 722}, {'text': 'Основатель "ВКонтакте" Павел Дуров рассказал об успехах Telegram .', 'entities': [['NAME', 23, 28], ['SURNAME', 29, 34]], 'source': 'factRuEval2016/testset/book_3573', 'source_start': 108}, {'text': 'Однако главы МИД не будут замыкаться на одном направлении, заявил депутат Милли Меджлиса Айдын Мирзазаде .', 'entities': [['NAME', 89, 94], ['SURNAME', 95, 104]], 'source': 'factRuEval2016/testset/book_3574', 'source_start': 188}, {'text': 'Моне сделал для Франции, её самоописания и самоощущения, хоронящегося внутри мощных исторических потоков и порывов, столько же, сколько сделал для формирования российской литературы и общественной мысли Лев Толстой.', 'entities': [['SURNAME', 0, 4], ['NAME', 203, 206], ['SURNAME', 207, 214]], 'source': 'factRuEval2016/testset/book_3581', 'source_start': 99}, {'text': 'Среди них академики Владимир Захаров, Леонид Келдыш, Александр Кулешов, Валерий Рубаков, Сергей Стишов, Виктор Васильев, Николай Кардашев, Алексей Старобинский, Роберт Сурис и Александр Чаплик.', 'entities': [['NAME', 20, 28], ['SURNAME', 29, 36], ['NAME', 38, 44], ['SURNAME', 45, 51], ['NAME', 53, 62], ['SURNAME', 63, 70], ['NAME', 72, 79], ['SURNAME', 80, 87], ['NAME', 89, 95], ['SURNAME', 96, 102], ['NAME', 104, 110], ['SURNAME', 111, 119], ['NAME', 121, 128], ['SURNAME', 129, 137], ['NAME', 139, 146], ['SURNAME', 147, 159], ['NAME', 161, 167], ['SURNAME', 168, 173], ['NAME', 176, 185], ['SURNAME', 186, 192]], 'source': 'factRuEval2016/testset/book_3602', 'source_start': 422}, {'text': 'Вернувшись со службы, где ведает смертями и браками, Воробьянинов застаёт двух посторонних людей: священника церкви Фрола и Лавра отца Фёдора и соседку, агрономшу Кузнецову.', 'entities': [['SURNAME', 53, 65], ['NAME', 116, 121], ['NAME', 124, 129], ['NAME', 135, 141], ['SURNAME', 163, 172]], 'source': 'factRuEval2016/testset/book_3615', 'source_start': 1916}, {'text': 'Кремль опроверг новые обвинения Алексея Навального о связях между строителями так называемого дворца Путина под Геленджиком и администрацией главы государства.', 'entities': [['NAME', 32, 39], ['SURNAME', 40, 50], ['SURNAME', 101, 107]], 'source': 'factRuEval2016/testset/book_3632', 'source_start': 145}, {'text': 'Об этом заявил замглавы российского МИДа Григорий Карасин.', 'entities': [['NAME', 41, 49], ['SURNAME', 50, 57]], 'source': 'factRuEval2016/testset/book_3644', 'source_start': 278}, {'text': 'Владимир Путин потребовал разработать "эффективную модель" функционирования пригородного железнодорожного транспорта.', 'entities': [['NAME', 0, 8], ['SURNAME', 9, 14]], 'source': 'factRuEval2016/testset/book_3667', 'source_start': 157}, {'text': 'Этот доклад — реакция на доклад американского социального психолога Филипа Зимбардо (того самого, что провел знаменитый Стэнфордский тюремный эксперимент) «Исчезновение мужчин».', 'entities': [['NAME', 68, 74], ['SURNAME', 75, 83]], 'source': 'factRuEval2016/testset/book_3677', 'source_start': 389}, {'text': 'Правообладатели связывают перемену в отношении руководства соцсети к пиратскому контенту с тем, что Павел Дуров покинул пост ее генерального директора.', 'entities': [['NAME', 100, 105], ['SURNAME', 106, 111]], 'source': 'factRuEval2016/testset/book_3680', 'source_start': 47}, {'text': 'Мы верим, что вместе с партнерами из СИБУРа мы сможем принести людям радость и подарить им интересные впечатления от встречи с театром», отметил художественный руководитель Театра Наций Евгений Миронов.', 'entities': [['NAME', 186, 193], ['SURNAME', 194, 201]], 'source': 'factRuEval2016/testset/book_3687', 'source_start': 2431}, {'text': 'Во вторник президент Владимир Путин должен обсудить регулирование Рунета с крупнейшими интернет-компаниями.', 'entities': [['NAME', 21, 29], ['SURNAME', 30, 35]], 'source': 'factRuEval2016/testset/book_3688', 'source_start': 42}, {'text': 'При этом он отказался уточнить, какие именно вопросы хочет обсудить с врио мэра Москвы Сергеем Собяниным.', 'entities': [['NAME', 87, 94], ['SURNAME', 95, 104]], 'source': 'factRuEval2016/testset/book_3696', 'source_start': 628}, {'text': 'Контроль над компанией возвращается к Михаилу Гуцериеву', 'entities': [['NAME', 38, 45], ['SURNAME', 46, 55]], 'source': 'factRuEval2016/testset/book_3697', 'source_start': 31}, {'text': 'В этом фрагменте Владимир Путин говорит о том, как началась работа «по возврату Крыма в состав России».', 'entities': [['NAME', 17, 25], ['SURNAME', 26, 31]], 'source': 'factRuEval2016/testset/book_3700', 'source_start': 143}, {'text': 'Точки зрения президентов двух стран о возможных путях выхода из тяжелой кризисной ситуации в значительной степени совпадают», — сказал пресс-секретарь Путина Дмитрий Песков.', 'entities': [['SURNAME', 151, 157], ['NAME', 158, 165], ['SURNAME', 166, 172]], 'source': 'factRuEval2016/testset/book_3702', 'source_start': 952}, {'text': 'Основатель Regnum сообщил о продаже агентства холдингу «Газпром-медиа» Новым руководителем агентства Regnum станет Юлия Крижанская.', 'entities': [['NAME', 115, 119], ['SURNAME', 120, 130]], 'source': 'factRuEval2016/testset/book_3708', 'source_start': 0}, {'text': 'Канцлер ФРГ Ангела Меркель в преддверии открывающегося в Баварии саммита ведущих индустриальных стран Запада заявила, что участие России во встречах "большой семерки" в настоящее время немыслимо.', 'entities': [['NAME', 12, 18], ['SURNAME', 19, 26]], 'source': 'factRuEval2016/testset/book_3714', 'source_start': 93}, {'text': 'Президент Украины Петр Порошенко заявил, что полноценная нормализация отношений с Россией не будет осуществлена без возвращения Крыма.', 'entities': [['NAME', 18, 22], ['SURNAME', 23, 32]], 'source': 'factRuEval2016/testset/book_3716', 'source_start': 138}, {'text': 'Никаких комментариев о дальнейшей судьбе «MTV» в России не дал и президент информационно-издательского холдинга Николай Картозия.', 'entities': [['NAME', 112, 119], ['SURNAME', 120, 128]], 'source': 'factRuEval2016/testset/book_3725', 'source_start': 589}, {'text': 'С таким официальным заявлением в Ново-Огарево выступил президент России Владимир Путин, сообщает РБК.', 'entities': [['NAME', 72, 80], ['SURNAME', 81, 86]], 'source': 'factRuEval2016/testset/book_3734', 'source_start': 238}, {'text': 'Тем временем участницы панк-группы встретились с Хиллари Клинтон.', 'entities': [['NAME', 49, 56], ['SURNAME', 57, 64]], 'source': 'factRuEval2016/testset/book_3744', 'source_start': 0}, {'text': 'Президент Филип Вуянович при этом подчеркнул, что этот шаг вовсе не является демонстрацией антироссийских настроений.', 'entities': [['NAME', 10, 15], ['SURNAME', 16, 24]], 'source': 'factRuEval2016/testset/book_3746', 'source_start': 61}, {'text': 'В полемику вступили президент Ближневосточного форума Дэниэл Пайпс и специалист по демографии Марк Адоманис.', 'entities': [['NAME', 54, 60], ['SURNAME', 61, 66], ['NAME', 94, 98], ['SURNAME', 99, 107]], 'source': 'factRuEval2016/testset/book_3752', 'source_start': 67}, {'text': 'Идею одной яркой и беспрецедентной для России арт-выставки привёз из Швеции художник Максим Илюхин.', 'entities': [['NAME', 85, 91], ['SURNAME', 92, 98]], 'source': 'factRuEval2016/testset/book_3756', 'source_start': 74}, {'text': 'Оппозиционер предложил западным странам нанести «серьезный удар» по «ближнему кругу» российского президента Владимира Путина в ответ на присоединение Крыма.', 'entities': [['NAME', 108, 117], ['SURNAME', 118, 124]], 'source': 'factRuEval2016/testset/book_3757', 'source_start': 55}, {'text': 'Американской делегации во главе с замминистра сельского хозяйства США Джеймсом Миллером предстоит урегулировать этот хлорный конфликт.', 'entities': [['NAME', 70, 78], ['SURNAME', 79, 87]], 'source': 'factRuEval2016/testset/book_3760', 'source_start': 644}, {'text': 'Остальную половину нужно выплачивать из собственных ресурсов или привлекать средства на рынке», — рассказывает заместитель директора группы государственных финансов S&P Карен Вартапетов.', 'entities': [['NAME', 169, 174], ['SURNAME', 175, 185]], 'source': 'factRuEval2016/testset/book_3762', 'source_start': 3664}, {'text': 'Дмитрий Медведев объяснил, почему это выгодно для России.', 'entities': [['NAME', 0, 7], ['SURNAME', 8, 16]], 'source': 'factRuEval2016/testset/book_3764', 'source_start': 66}, {'text': 'Об этом рассказал гендиректор ГАБТа Владимир Урин, уточнив, что эту практику руководство Большого переняло у западных театров.', 'entities': [['NAME', 36, 44], ['SURNAME', 45, 49]], 'source': 'factRuEval2016/testset/book_3767', 'source_start': 127}, {'text': 'Глава президентского Совета по правам человека (СПЧ) Михаил Федотов выступил с инициативой предусмотреть в законодательстве вопрос о саморегулировании в сфере средств массовой информации.', 'entities': [['NAME', 53, 59], ['SURNAME', 60, 67]], 'source': 'factRuEval2016/testset/book_3768', 'source_start': 197}]
cases.extend(corpus_cases)
test_rng.shuffle(cases)
counts=Counter(lab for c in cases for lab,_,_ in c["entities"])
assert len(cases)>=100 and all(counts[lab]>=50 for lab in LABELS)
for c in cases: normalize_entities(c["text"],c["entities"])
print("Всего текстов:",len(cases),"; фрагментов factRuEval:",len(corpus_cases))
print("Поддержка классов:",dict(counts))

## 10. Метрики качества распознавания
Отдельно считаем F1 целых сущностей (entity-level) (тип и символьные границы полностью совпадают) и F1 BIO-разметки токенов (token-level BIO) на общей токенизации RuBERT. `TP` — совпадения, `FP` — лишние, `FN` — пропуски. Precision=TP/(TP+FP), Recall=TP/(TP+FN), F1=2PR/(P+R). Macro F1 — среднее по 10 классам, Weighted F1 — среднее с весами support. `O` не входит в средние, чтобы доля обычных слов не завышала оценку. В BIO-оценке ошибка B/I при правильном классе тоже считается FP+FN. Если один токен пересекает две сущности, выбирается максимальное перекрытие (при равенстве — первая по позиции); число таких случаев выводится. Оценка целевой системы исключительно по десяти обязательным классам.

In [ ]:
def text_to_bio(text,entities):
    tokens=person_ner.tokenizer(text,add_special_tokens=False,return_offsets_mapping=True,truncation=False)
    offsets=tokens['offset_mapping']; ids=tokens['input_ids']
    out=[]; previous=None; conflicts=0
    for s,e in offsets:
        hits=[(min(e,b)-max(s,a),idx,lab) for idx,(lab,a,b) in enumerate(entities) if s<b and a<e]
        if len(hits)>1: conflicts+=1
        if not hits:
            out.append('O'); previous=None; continue
        _,idx,lab=max(hits,key=lambda x:(x[0],-x[1]))
        out.append(('I-' if previous==idx else 'B-')+lab); previous=idx
    return person_ner.tokenizer.convert_ids_to_tokens(ids),out,offsets,conflicts

def score_sets(golds,preds):
    stats={l:Counter(TP=0,FP=0,FN=0) for l in LABELS}
    for gold,pred in zip(golds,preds):
        for lab in LABELS:
            g={x for x in gold if x[0]==lab}; p={x for x in pred if x[0]==lab}
            stats[lab].update(TP=len(g&p),FP=len(p-g),FN=len(g-p))
    rows=[]
    for lab in LABELS:
        tp,fp,fn=(stats[lab][x] for x in ('TP','FP','FN'))
        p=tp/(tp+fp) if tp+fp else 0; r=tp/(tp+fn) if tp+fn else 0
        f=2*p*r/(p+r) if p+r else 0
        rows.append(dict(label=lab,precision=p,recall=r,f1=f,support=tp+fn,TP=tp,FP=fp,FN=fn))
    df=pd.DataFrame(rows).set_index('label')
    for avg in ('macro','weighted'):
        weight=df.loc[LABELS,'support'] if avg=='weighted' else pd.Series(1,index=LABELS)
        df.loc[avg,['precision','recall','f1']]=df.loc[LABELS,['precision','recall','f1']].mul(weight,axis=0).sum()/weight.sum()
        df.loc[avg,'support']=df.loc[LABELS,'support'].sum()
    return df

def bio_set(tags): return {(tag[2:],i,tag[:1]) for i,tag in enumerate(tags) if tag!='O'}

texts=[c['text'] for c in cases]
predictions=[]
for start in range(0,len(texts),16):
    batch=texts[start:start+16]
    raw=person_ner(batch,batch_size=16)
    predictions.extend(recognize_pdn(t,raw_ner=r) for t,r in zip(batch,raw))
assert len(predictions)==len(cases)
entity_gold=[set(map(tuple,c['entities'])) for c in cases]
entity_pred=[set(map(tuple,p)) for p in predictions]
entity_metrics=score_sets(entity_gold,entity_pred)
bio_gold=[]; bio_pred=[]; conflicts=0
for c,p in zip(cases,predictions):
    _,g,_,cg=text_to_bio(c['text'],c['entities']); _,b,_,cp=text_to_bio(c['text'],p)
    bio_gold.append(bio_set(g)); bio_pred.append(bio_set(b)); conflicts+=cg+cp
bio_metrics=score_sets(bio_gold,bio_pred)
print('ENTITY-LEVEL:'); display(entity_metrics.round(4))
print('TOKEN-LEVEL BIO:'); display(bio_metrics.round(4))
print('Пересечения двух сущностей с одним токеном (эталон+прогноз):',conflicts)
for title,df in [('entity',entity_metrics),('BIO',bio_metrics)]:
    print(title,'требования выполнены:', bool(df.loc['macro','f1']>=.8 and df.loc['weighted','f1']>=.85))
errors_ner=[{'text':c['text'],'expected':c['entities'],'predicted':p,'source':c['source']} for c,p,g,h in zip(cases,predictions,entity_gold,entity_pred) if g!=h]
print('Текстов с несовпадениями:',len(errors_ner)); print(*errors_ner[:8],sep='\n')
# Дополнительные раздельные метрики: синтетика и фактRuEval.
for src in ('synthetic','factRuEval'):
    idx=[i for i,c in enumerate(cases) if c['source'].startswith(src)]
    subset=score_sets([entity_gold[i] for i in idx],[entity_pred[i] for i in idx])
    print(src,'текстов:',len(idx)); display(subset.loc[[l for l in LABELS if subset.loc[l,'support']>0],['precision','recall','f1','support']].round(4))

## 11. Сохранение свойств и BERTScore


In [ ]:
from bert_score import score as bert_score
shared_memo={}; paired=[]; anon_errors=[]; morphology_audit=[]
for i,c in enumerate(cases):
    local_errors=[]; local_audit=[]
    syn=anonymize(c['text'],c['entities'],memo=shared_memo,on_error='redact',errors=local_errors,audit=local_audit)
    masked=mask(c['text'],c['entities'])
    paired.append({'text':c['text'],'synonym':syn,'masked':masked,'has_entities':bool(c['entities']),
                   'strict_success':not local_errors,'source':c['source']})
    anon_errors.extend({'case':i,**e} for e in local_errors)
    morphology_audit.extend({'case_index':i,**e} for e in local_audit)
print('Ошибок синонимической замены:',len(anon_errors))
print('Текстов с резервным маскированием:',sum(not x['strict_success'] for x in paired))
print('Примеры ошибок:',anon_errors[:5])
print('Успешных резервных морфологических замен:',len(morphology_audit))
print('Примеры резервного анализа:',morphology_audit[:5])
refs=[x['text'] for x in paired]
bert_kwargs=dict(model_type='bert-base-multilingual-cased',num_layers=9,batch_size=16,
                 device='cuda' if torch.cuda.is_available() else 'cpu',verbose=True,rescale_with_baseline=False)
for method in ('synonym','masked'):
    p,r,f=bert_score([x[method] for x in paired],refs,**bert_kwargs)
    for row,a,b,c in zip(paired,p.tolist(),r.tolist(),f.tolist()):
        row.update({method+'_P':a,method+'_R':b,method+'_F1':c})
bert_df=pd.DataFrame(paired)
summary=[]
for subset,df in [('all',bert_df),('with_entities',bert_df[bert_df.has_entities]),
                  ('strict_success_with_entities',bert_df[bert_df.has_entities & bert_df.strict_success])]:
    if df.empty: continue
    for method in ('synonym','masked'):
        summary.append({'subset':subset,'method':method,'texts':len(df),
                        **{m:df[method+'_'+m].mean() for m in ('P','R','F1')}})
bert_summary=pd.DataFrame(summary)
display(bert_summary.round(5))
valid=bert_df[bert_df.has_entities & bert_df.strict_success]
if not valid.empty:
    delta=(valid.synonym_F1-valid.masked_F1).mean()
    print('Разность средней F1 (синонимический метод − маскирование):',round(delta,5))
    print('Гипотеза о более высоком BERTScore синонимической замены:', 'поддержана' if delta>0 else 'не поддержана')
else: print('Нет текстов с успешной строгой заменой; вывод невозможен')
# Дополнительно: число проблем на выходе алгоритма при использовании предсказанных сущностей.
end_to_end_errors=[]
for i,(c,pred) in enumerate(zip(cases,predictions)):
    issues=[]
    anonymize(c['text'],pred,on_error='redact',errors=issues)
    end_to_end_errors.extend({'case':i,**x} for x in issues)
print('Ошибок замены при фактических предсказаниях:',len(end_to_end_errors))
print('Пропущенные при распознавании сущности (строго):',sum(len(g-p) for g,p in zip(entity_gold,entity_pred)))

## 12. Демонстрация всей системы и экспорт результатов

Модель загружается из релиза GitHub. Итоговый архив результатов можно скачать через панель файлов Colab.


In [ ]:
demo='Имя: Иван. Фамилия: Петров. '+ ' '.join(field_names[l]+': '+structured_example(l,7)+'.' for l in field_names)
new,detected=anon(demo,on_error='redact')
print('Исходный:',demo)
print('Найдено:',[(l,demo[s:e],s,e) for l,s,e in detected])
print('Обезличено:',new)
print('Второй вызов:',anon(demo,on_error='redact')[0])
my_text='Саша пришла на работу. E-mail: sample@example.org.'
print(anon(my_text,on_error='redact'))

In [ ]:
OUT=Path('/content/KR22_results'); OUT.mkdir(exist_ok=True)
for filename,obj in [('test_cases.json',cases),('recognition_errors.json',errors_ner),('anonymization_errors.json',anon_errors),('morphology_audit.json',morphology_audit),('morphology_regression.json',regression_results),('paired_texts.json',paired),('end_to_end_errors.json',end_to_end_errors)]:
    (OUT/filename).write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf-8')
entity_metrics.to_csv(OUT/'entity_metrics.csv'); bio_metrics.to_csv(OUT/'bio_metrics.csv')
bert_summary.to_csv(OUT/'bertscore_summary.csv',index=False)
bert_df.to_csv(OUT/'bertscore_per_text.csv',index=False)
import shutil
archive=shutil.make_archive('/content/KR22_results','zip',OUT)
print('Результаты для отчёта:',archive)